# Exact Fig. 5 reproduction from manuscript aggregate inputs

All analysis and plotting cells are retained from the manuscript notebook. Only input and output paths are changed.


## Portable path configuration
This release copy contains no personal or NAS paths. Set `GAO2026_DATA_ROOT` for deposited data and `GAO2026_RAW_ROOT` only when running raw-image stages (raw revision microscopy is not included).


In [ ]:
from pathlib import Path
import os
_default_repo_root = Path.cwd().resolve()
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', _default_repo_root / 'data')).expanduser().resolve()
GAO2026_RAW_ROOT = Path(os.environ.get('GAO2026_RAW_ROOT', GAO2026_DATA_ROOT / 'external_raw')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', _default_repo_root / 'outputs')).expanduser().resolve()
GAO2026_EXTERNAL_ROOT = Path(os.environ.get('GAO2026_EXTERNAL_ROOT', GAO2026_DATA_ROOT / 'external')).expanduser().resolve()


# Integrated plotting notebook: HDAC inhibitors (Nanog + Sox2)

This notebook reuses pre-computed CSV outputs from:

- `HDAC_inhibitor/Nanog/inhibitor_response_analysis-0909_hov1.ipynb`
- `HDAC_inhibitor/Sox2/inhibitor_response_analysis-0908_hov1.ipynb`

and generates candidate panels for **Fig. 5 (main)** and **Extended Data (ED)**.

## Expected folder structure

This notebook should be placed at:

`HDAC_inhibitor/integrated/figure5_integrated.ipynb`

and assumes the following relative paths:

- `../Nanog/tetr_mcp_hybrid_analysis/`
- `../Sox2/tetr_mcp_hybrid_analysis/`

Each `tetr_mcp_hybrid_analysis/` folder should contain (at minimum):

- `aggregated_perframe.csv`  (frame-level ON/OFF)
- `aggregated_runs_by_cell.csv` (run-level ON/OFF with `cell_uid`; preferred)
  - if missing, the notebook falls back to `aggregated_runs.csv`

Outputs are saved under:

- `./figs/` (PDF)
- `./tables/` (CSV)

> **Font**: all plots are configured to use **Arial** (with fallback if Arial is unavailable on the system).


In [ ]:
# --- Imports & global plot style (Arial, PDF) ---

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# (optional) stats utilities
from scipy.stats import chi2

# ---- Output folders ----
FIGDIR = GAO2026_OUTPUT_ROOT / "12_Fig5_integrated/figs"
TABLEDIR = GAO2026_OUTPUT_ROOT / "12_Fig5_integrated/tables"
FIGDIR.mkdir(exist_ok=True, parents=True)
TABLEDIR.mkdir(exist_ok=True, parents=True)

# ---- Global plotting parameters (edit here) ----
PLOT_CFG = {
    # font sizes
    "font_base": 8,
    "font_axes_label": 9,
    "font_axes_title": 9,
    "font_ticks": 8,
    "font_legend": 7,

    # figure sizes (inches)
    "figsize_small": (3.3, 2.6),
    "figsize_medium": (4.0, 3.0),
    "figsize_wide": (6.0, 3.0),
    "dpi": 300,
}

def set_mpl_style(cfg=PLOT_CFG):
    """Set matplotlib rcParams (Arial + PDF text embedding)."""
    plt.rcParams.update({
        # font
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": cfg["font_base"],
        "axes.labelsize": cfg["font_axes_label"],
        "axes.titlesize": cfg["font_axes_title"],
        "xtick.labelsize": cfg["font_ticks"],
        "ytick.labelsize": cfg["font_ticks"],
        "legend.fontsize": cfg["font_legend"],

        # publication-friendly PDF/PS fonts (editable text)
        "pdf.fonttype": 42,
        "ps.fonttype": 42,

        # simple look
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

set_mpl_style()
print("[OK] matplotlib style set. Output:", FIGDIR.resolve())


In [ ]:
# --- Paths to the exact aggregate inputs used by the manuscript notebook ---

ROOT = (GAO2026_DATA_ROOT / "12_Fig5_integrated").resolve()
NANOG_DIR = (ROOT / "Nanog" / "tetr_mcp_hybrid_analysis").resolve()
SOX2_DIR = (ROOT / "Sox2" / "tetr_mcp_hybrid_analysis").resolve()

print("NANOG_DIR:", NANOG_DIR)
print("SOX2_DIR :", SOX2_DIR)

for path in (NANOG_DIR, SOX2_DIR):
    if not path.exists():
        raise FileNotFoundError(path)


In [ ]:
# --- Data loader (reuses aggregated CSVs) ---

def _ensure_cell_uid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "cell_uid" not in df.columns:
        if ("file" in df.columns) and ("cell_id" in df.columns):
            df["cell_uid"] = df["file"].astype(str) + "_c" + df["cell_id"].astype(str)
        else:
            raise ValueError("Cannot construct cell_uid (need columns: file, cell_id).")
    return df

def load_dataset(gene: str, analysis_dir: Path) -> dict:
    analysis_dir = Path(analysis_dir)

    perframe_path = analysis_dir / "aggregated_perframe.csv"
    runs_by_cell_path = analysis_dir / "aggregated_runs_by_cell.csv"
    runs_path = analysis_dir / "aggregated_runs.csv"

    if not perframe_path.exists():
        raise FileNotFoundError(perframe_path)

    perframe = pd.read_csv(perframe_path)
    perframe = _ensure_cell_uid(perframe)

    if runs_by_cell_path.exists():
        runs = pd.read_csv(runs_by_cell_path)
    elif runs_path.exists():
        runs = pd.read_csv(runs_path)
    else:
        raise FileNotFoundError(f"Neither {runs_by_cell_path.name} nor {runs_path.name} exists in {analysis_dir}")

    runs = _ensure_cell_uid(runs)

    # Basic sanity checks
    req_runs = {"group","kind","length_min","left_censored","right_censored","cell_uid"}
    missing = req_runs - set(runs.columns)
    if missing:
        raise ValueError(f"Missing columns in runs: {missing}")

    req_pf = {"group","frame","on_off","cell_uid"}
    missing_pf = req_pf - set(perframe.columns)
    if missing_pf:
        raise ValueError(f"Missing columns in perframe: {missing_pf}")

    return {
        "gene": gene,
        "dir": analysis_dir,
        "perframe": perframe,
        "runs": runs,
    }

datasets = []
for gene, p in [("Nanog", NANOG_DIR), ("Sox2", SOX2_DIR)]:
    try:
        datasets.append(load_dataset(gene, p))
        print(f"[OK] Loaded {gene}: perframe={len(datasets[-1]['perframe'])}, runs={len(datasets[-1]['runs'])}")
    except Exception as e:
        print(f"[SKIP] {gene}: {e}")

datasets


In [ ]:
# --- Group labels / ordering ---

# Minimal (main-figure) set
GROUP_ORDER_MAIN = ["DMSO", "TSA", "RGFP966_0h"]

# Full set (ED)
GROUP_ORDER_FULL = ["DMSO", "TSA", "RGFP966_0h"]

DISPLAY_NAME = {
    "RGFP966_0h": "RGFP966",
    "RGFP966_1h": "RGFP966 (1h)",
}


KIND_LABEL = {
    "ON": "Active",
    "OFF": "Inactive"
}

def nice_group(g: str) -> str:
    return DISPLAY_NAME.get(g, g)


In [ ]:
# --- Survival analysis utilities (KM, RMST, cluster bootstrap) ---

def kaplan_meier(times, events):
    """Kaplan–Meier estimator for right-censored data.

    Parameters
    ----------
    times : array-like (>=0)
    events : array-like (1=event observed, 0=right-censored)

    Returns
    -------
    t : np.ndarray
        Step times, starting with 0.
    S : np.ndarray
        Survival values S(t)=P(T>=t) in 'post' convention.
    """
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)

    m = np.isfinite(times) & (times >= 0) & np.isfinite(events)
    times = times[m]; events = events[m]

    if len(times) == 0:
        return np.array([0.0]), np.array([1.0])

    event_times = np.unique(times[events == 1])
    S = 1.0
    t_list = [0.0]
    s_list = [1.0]

    for t in event_times:
        n_risk = np.sum(times >= t)
        d = np.sum((times == t) & (events == 1))
        if n_risk <= 0:
            continue
        S *= (1.0 - d / n_risk)
        t_list.append(float(t))
        s_list.append(float(S))

    return np.array(t_list), np.array(s_list)

def rmst_from_km(t, S, tau):
    """Restricted mean survival time (RMST) = ∫_0^tau S(u) du for a KM step function."""
    t = np.asarray(t, float)
    S = np.asarray(S, float)
    tau = float(tau)

    if tau <= 0:
        return 0.0

    # ensure sorted
    order = np.argsort(t)
    t = t[order]; S = S[order]

    area = 0.0
    for i in range(len(t)):
        t_i = t[i]
        if t_i >= tau:
            break
        t_next = t[i+1] if i+1 < len(t) else tau
        t_next = min(t_next, tau)
        if t_next > t_i:
            area += S[i] * (t_next - t_i)
    return float(area)

def prepare_runs(runs: pd.DataFrame, kind: str, groups, exclude_left=True):
    df = runs[(runs["kind"] == kind) & (runs["group"].isin(groups))].copy()
    if exclude_left:
        df = df[~df["left_censored"]].copy()
    df["event"] = (~df["right_censored"]).astype(int)
    df = df[np.isfinite(df["length_min"]) & (df["length_min"] > 0)].copy()
    return df

def rmst_table(runs: pd.DataFrame, kind: str, groups, tau=30.0, exclude_left=True) -> pd.DataFrame:
    df = prepare_runs(runs, kind=kind, groups=groups, exclude_left=exclude_left)
    rows = []
    for grp, sub in df.groupby("group", sort=False):
        t, S = kaplan_meier(sub["length_min"].values, sub["event"].values)
        rows.append({
            "group": grp,
            f"RMST_{kind}_tau{int(tau)}min": rmst_from_km(t, S, tau=tau),
            "n_runs": len(sub),
            "n_cells": sub["cell_uid"].nunique(),
        })
    out = pd.DataFrame(rows)
    # add delta vs DMSO if present
    if "DMSO" in out["group"].values:
        base = float(out.loc[out["group"]=="DMSO", f"RMST_{kind}_tau{int(tau)}min"].iloc[0])
        out["Delta_vs_DMSO"] = out[f"RMST_{kind}_tau{int(tau)}min"] - base
    return out

def cluster_bootstrap_rmst_delta(
    runs: pd.DataFrame,
    kind: str,
    tau: float,
    base: str,
    groups,
    B: int = 2000,
    seed: int = 0,
    exclude_left: bool = True,
) -> pd.DataFrame:
    """Cell-cluster bootstrap for ΔRMST (group - base).

    Resample cells (cell_uid) with replacement within each group, keep all runs for sampled cells.
    """
    rng = np.random.default_rng(seed)
    df = prepare_runs(runs, kind=kind, groups=groups, exclude_left=exclude_left)

    # pre-split by group -> dict(cell_uid -> (t, event))
    cell_runs = {}
    cell_list = {}
    for grp, sub_g in df.groupby("group"):
        d = {}
        for cid, sub_c in sub_g.groupby("cell_uid"):
            d[cid] = (sub_c["length_min"].values.astype(float), sub_c["event"].values.astype(int))
        cell_runs[grp] = d
        cell_list[grp] = np.array(list(d.keys()))

    if base not in cell_runs:
        raise ValueError(f"base group '{base}' not found in data")

    def _rmst_from_sample(grp, sampled_cells):
        # concatenate runs for sampled cells (duplicates allowed)
        t_all = []
        e_all = []
        d = cell_runs[grp]
        for cid in sampled_cells:
            t, e = d[cid]
            t_all.append(t)
            e_all.append(e)
        t_all = np.concatenate(t_all) if len(t_all) else np.array([])
        e_all = np.concatenate(e_all) if len(e_all) else np.array([])
        tk, S = kaplan_meier(t_all, e_all)
        return rmst_from_km(tk, S, tau=tau)

    # observed RMSTs
    obs = rmst_table(runs, kind=kind, groups=groups, tau=tau, exclude_left=exclude_left)
    obs_map = {r["group"]: r[f"RMST_{kind}_tau{int(tau)}min"] for r in obs.to_dict("records")}
    base_obs = obs_map[base]

    out_rows = []
    for grp in groups:
        if grp == base:
            continue
        if grp not in cell_runs:
            continue

        deltas = np.empty(B, float)
        for b in range(B):
            base_cells = rng.choice(cell_list[base], size=len(cell_list[base]), replace=True)
            grp_cells  = rng.choice(cell_list[grp],  size=len(cell_list[grp]),  replace=True)
            rmst_base = _rmst_from_sample(base, base_cells)
            rmst_grp  = _rmst_from_sample(grp,  grp_cells)
            deltas[b] = rmst_grp - rmst_base

        delta_obs = float(obs_map[grp] - base_obs)
        lo, hi = np.percentile(deltas, [2.5, 97.5])

        # 'p-value like' (two-sided, sign test on bootstrap distribution)
        p_two = 2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0))
        p_two = float(min(1.0, p_two))

        out_rows.append({
            "kind": kind,
            "tau_min": tau,
            "compare": f"{base} vs {grp}",
            "Delta_obs(grp-base)": delta_obs,
            "CI95_low": float(lo),
            "CI95_high": float(hi),
            "p_boot_two_sided": p_two,
            "B": B,
            "n_cells_base": int(len(cell_list[base])),
            "n_cells_grp": int(len(cell_list[grp])),
        })

    return pd.DataFrame(out_rows)


In [ ]:
# --- Plot helpers (save as PDF) ---

def savefig_pdf(fig, outpath: Path):
    outpath = Path(outpath)
    outpath.parent.mkdir(exist_ok=True, parents=True)
    fig.savefig(outpath, bbox_inches="tight", dpi=PLOT_CFG["dpi"])
    plt.close(fig)
    print("[saved]", outpath)

def plot_km_panel(runs: pd.DataFrame, kind: str, gene: str, groups, exclude_left=True,
                  logy=True, xlim=None, ylim=None, title=None, outname=None):

    df = prepare_runs(runs, kind=kind, groups=groups, exclude_left=exclude_left)

    fig, ax = plt.subplots(figsize=(3.33,2.5))

    for grp in groups:
        sub = df[df["group"] == grp]
        if sub.empty:
            continue
        t, S = kaplan_meier(sub["length_min"].values, sub["event"].values)
        ax.step(t, S, where="post",
                label=f"{nice_group(grp)} (runs={len(sub)})")

    # ---- 表示用ラベルを変換 ----
    kind_label = KIND_LABEL.get(kind, kind)

    ax.set_xlabel(f"{kind_label} duration (min)")
    ax.set_ylabel("Survival S(t) = P(duration ≥ t)")

    if title is None:
        title = f"{gene}: Kaplan–Meier ({kind_label})"
    ax.set_title(title)

    if logy:
        ax.set_yscale("log")
        if ylim is None:
            ax.set_ylim(1e-3, 1.05)
    else:
        ax.set_ylim(0, 1.05)

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.grid(alpha=0.3)
    ax.legend(frameon=False)

    if outname is None:
        outname = f"{gene}_KM_{kind_label}_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)






from scipy.stats import mannwhitneyu
import numpy as np

def holm_adjust_array(pvals):
    """Holm step-down adjustment for a list/array of p-values."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)

    running_max = 0.0
    for rank, idx in enumerate(order):
        adj_p = (m - rank) * pvals[idx]
        running_max = max(running_max, adj_p)
        adj[idx] = min(1.0, running_max)
    return adj

def p_to_stars(p):
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return "ns"

def add_sig_bracket(ax, x1, x2, y, text, h=0.02, lw=0.8):
    """
    Draw a significance bracket between x1 and x2 at height y (data coords).
    h: vertical height of bracket (data coords)
    """
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color="black", lw=lw, clip_on=False)
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=PLOT_CFG.get("fontsize", 7))



def duty_cycle_pvals(dc: pd.DataFrame, base="DMSO", comps=("TSA","RGFP966_0h")):
    """
    dc: duty_cycle_table の出力（columns: group, cell_uid, duty_cycle）
    returns: list of dict with raw p and Holm-adjusted p
    """
    base_vals = dc.loc[dc["group"]==base, "duty_cycle"].dropna().values
    results = []

    raw_ps = []
    pairs = []
    for g in comps:
        vals = dc.loc[dc["group"]==g, "duty_cycle"].dropna().values
        if len(base_vals) < 2 or len(vals) < 2:
            p = np.nan
        else:
            p = mannwhitneyu(base_vals, vals, alternative="two-sided").pvalue
        raw_ps.append(p)
        pairs.append((base, g))

    adj_ps = holm_adjust_array(raw_ps)

    for (b, g), p, p_adj in zip(pairs, raw_ps, adj_ps):
        results.append({
            "compare": f"{b} vs {g}",
            "p_raw": float(p) if np.isfinite(p) else np.nan,
            "p_holm": float(p_adj) if np.isfinite(p_adj) else np.nan,
        })
    return results





def plot_rmst_delta(ci_df: pd.DataFrame, gene: str, outname=None, title=None):
    """Plot ΔRMST with 95% CI (from cluster bootstrap)."""
    if ci_df.empty:
        print("[WARN] Empty CI table")
        return

    fig, ax = plt.subplots(figsize=PLOT_CFG["figsize_medium"])

    comps = ci_df["compare"].tolist()
    # extract group name from 'DMSO vs X'
    groups = [c.split("vs")[1].strip() for c in comps]
    x = np.arange(len(groups))

    y = ci_df["Delta_obs(grp-base)"].values
    lo = ci_df["CI95_low"].values
    hi = ci_df["CI95_high"].values

    ax.errorbar(x, y, yerr=[y-lo, hi-y], fmt="o", capsize=3)
    ax.axhline(0, linestyle="--", linewidth=1)

    ax.set_xticks(x)
    ax.set_xticklabels([nice_group(g) for g in groups], rotation=30, ha="right")
    ax.set_ylabel(f"ΔRMST (min) vs {ci_df['compare'].iloc[0].split(' vs ')[0]}")
    if title is None:
        title = f"{gene}: ΔRMST of {ci_df['kind'].iloc[0]} (τ={int(ci_df['tau_min'].iloc[0])} min)"
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

    if outname is None:
        outname = f"{gene}_RMST_delta_tau{int(ci_df['tau_min'].iloc[0])}min.pdf"
    savefig_pdf(fig, FIGDIR / outname)

def plot_state_raster(perframe: pd.DataFrame, gene: str, groups, n_cells=35,
                      frame_to_min=1.0, outname=None, title=None):
    """ON/OFF raster (cells × time) for each group, stacked vertically."""
    pf = perframe[perframe["group"].isin(groups)].copy()
    pf = _ensure_cell_uid(pf)

    # build matrices per group
    mats = []
    for g in groups:
        sub = pf[pf["group"]==g].copy()
        if sub.empty:
            mats.append(None)
            continue

        mat = sub.pivot_table(index="cell_uid", columns="frame", values="on_off", aggfunc="first")
        # sort cells by duty cycle (descending)
        mat = mat.loc[mat.mean(axis=1).sort_values(ascending=False).index]
        mat = mat.iloc[:n_cells]
        mats.append(mat)

    # determine common x-axis
    max_frame = 0
    for m in mats:
        if m is not None and m.shape[1] > max_frame:
            max_frame = m.shape[1]
    t = np.arange(max_frame) * frame_to_min

    fig_h = 0.8*len(groups) + 1.2
    #fig, axes = plt.subplots(nrows=len(groups), ncols=1, figsize=(PLOT_CFG["figsize_wide"][0], fig_h), sharex=True)
    fig, axes = plt.subplots(nrows=len(groups), ncols=1, figsize=(2.0, 4.0), sharex=True)
    if len(groups) == 1:
        axes = [axes]

    for ax, g, mat in zip(axes, groups, mats):
        if mat is None:
            ax.axis("off")
            continue
        # reindex columns to full range
        mat2 = mat.reindex(columns=np.arange(max_frame))
        from matplotlib.colors import ListedColormap

        cmap = ListedColormap(["lightgray", "red"])  # OFF=0, ON=1
        img = ax.imshow(mat2.values, aspect="auto", interpolation="nearest",
                        vmin=0, vmax=1, cmap=cmap)
        
        # --- add horizontal white separators between cells (y only) ---
        n_rows = mat2.shape[0]
        ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)   # 行境界
        ax.grid(which="minor", axis="y", color="white", linewidth=0.3)
        ax.tick_params(which="minor", left=False, right=False)  # minor tickは表示しない
        
        #img = ax.imshow(mat2.values, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
        ax.set_ylabel(nice_group(g))
        #ax.set_yticks([])

    axes[-1].set_xlabel("Time (min)")
    # tick every 10 min
    if max_frame > 0:
        step = int(10/frame_to_min) if frame_to_min > 0 else 10
        xticks = np.arange(0, max_frame, step=max(1, step))
        axes[-1].set_xticks(xticks)
        axes[-1].set_xticklabels([str(int(x*frame_to_min)) for x in xticks])

    # if title is None:
    #     title = f"{gene}: ON/OFF state raster (top {n_cells} cells / group)"
    fig.suptitle(title, y=1.02)

    if outname is None:
        outname = f"{gene}_state_raster_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)


def duty_cycle_table(perframe: pd.DataFrame, groups) -> pd.DataFrame:
    """
    perframe: expects at least {group, cell_uid, on_off} in columns
    duty_cycle = mean on_off per cell (= fraction of ON frames)
    """
    pf = perframe[perframe["group"].isin(groups)].copy()
    pf = _ensure_cell_uid(pf)

    # Treat on_off as numeric (assumed 0/1)
    pf = pf[np.isfinite(pf["on_off"])].copy()

    dc = (pf.groupby(["group", "cell_uid"], sort=False)["on_off"]
            .mean()
            .reset_index()
            .rename(columns={"on_off": "duty_cycle"}))
    return dc

def plot_duty_cycle(perframe: pd.DataFrame, gene: str, groups, outname=None, title=None):
    dc = duty_cycle_table(perframe, groups)
    fig, ax = plt.subplots(figsize=(1.2,2.3))

    data = [dc.loc[dc["group"]==g, "duty_cycle"].values for g in groups]
    ax.boxplot(data, positions=np.arange(len(groups)), widths=0.5, showfliers=False)

    # overlay points (jitter)
    rng = np.random.default_rng(0)
    for i, g in enumerate(groups):
        y = dc.loc[dc["group"]==g, "duty_cycle"].values
        x = rng.normal(i, 0.06, size=len(y))
        ax.plot(x, y, "o", markersize=3, alpha=0.6)

    # -------------------- p-value annotations (MW + Holm) --------------------
    # Here, compare TSA/RGFP966 against DMSO as the base (only if present)
    base = "DMSO"
    comps = [g for g in ["TSA", "RGFP966"] if g in groups]

    if base in groups and len(comps) > 0:
        stats = duty_cycle_pvals(dc, base=base, comps=tuple(comps))

        # Bracket height: add a little above the data maximum
        y_max = np.nanmax(dc["duty_cycle"].values)
        y = y_max + 0.03  # Adjust here for appearance (duty is 0–1, so a fixed offset is OK)
        step = 0.12       # Stagger to avoid overlapping multiple brackets

        # Use the index within groups for x positions
        g_to_x = {g:i for i, g in enumerate(groups)}

        for k, r in enumerate(stats):
            comp = r["compare"]  # "DMSO vs TSA"
            g = comp.split(" vs ")[1]
            x1 = g_to_x[base]
            x2 = g_to_x[g]

            p_adj = r["p_holm"]
            # Display format: show p-values or stars
            label = f"{p_to_stars(p_adj)}" if np.isfinite(p_adj) else "n/a"
            # If you want to show p-values directly:
            # label = f"p={p_adj:.3g}" if np.isfinite(p_adj) else "n/a"

            add_sig_bracket(ax, x1, x2, y + k*step, label, h=0.015, lw=0.8)
            ax.set_xticks(np.arange(len(groups)))
            ax.set_xticklabels([nice_group(g) for g in groups], rotation=30, ha="right")
            ax.set_ylabel("Fraction of Active frames")
            if title is not None:
                ax.set_title(title)
            ax.grid(axis="y", alpha=0.3)

            if outname is None:
                outname = f"{gene}_duty_cycle_groups{'-'.join(groups)}.pdf"
            savefig_pdf(fig, FIGDIR / outname)
            return dc

## Candidate panels for Fig. 5 (main)

This section generates **simple, story-forward** panels:

- **State raster** (qualitative): ON/OFF over time for representative cells.
- **Duty cycle (cell-level)**: fraction of frames ON per cell.
- **ΔRMST of ON duration (τ=30 min)**: a censor-aware scalar summary of the KM curve, with **cell-cluster bootstrap 95% CI** and a two-sided *p*-value-like statistic.

To keep the main figure minimal, use `GROUP_ORDER_MAIN = ["DMSO","TSA","RGFP966_0h"]`.
(Full 4-group plots including `RGFP966_1h` are generated in the ED section.)


In [ ]:
# --- Fig. 5 main candidates: raster, duty cycle, RMST (ON) ---

TAU = 30.0
BOOT_B = 2000
SEED = 0

for ds in datasets:
    gene = ds["gene"]
    runs = ds["runs"]
    perframe = ds["perframe"]

    print("\n=== ", gene, "===")

    # (1) state raster
    plot_state_raster(perframe, gene=gene, groups=GROUP_ORDER_MAIN, n_cells=100,
                      outname=f"Fig5_{gene}_state_raster_MAIN.pdf")

    # (2) duty cycle
    dc = plot_duty_cycle(perframe, gene=gene, groups=GROUP_ORDER_MAIN,
                         outname=f"Fig5_{gene}_duty_cycle_MAIN.pdf")
    dc.to_csv(TABLEDIR / f"{gene}_duty_cycle_celllevel_MAIN.csv", index=False)

    # (3) RMST summary table (ON)
    rmst = rmst_table(runs, kind="ON", groups=GROUP_ORDER_MAIN, tau=TAU, exclude_left=True)
    display(rmst)
    rmst.to_csv(TABLEDIR / f"{gene}_RMST_ON_tau{int(TAU)}min_MAIN.csv", index=False)

    # (4) cluster bootstrap CI for ΔRMST (ON)
    ci = cluster_bootstrap_rmst_delta(
        runs, kind="ON", tau=TAU, base="DMSO",
        groups=GROUP_ORDER_MAIN, B=BOOT_B, seed=SEED, exclude_left=True
    )
    display(ci)
    ci.to_csv(TABLEDIR / f"{gene}_RMST_ON_tau{int(TAU)}min_clusterBootstrap_MAIN.csv", index=False)
    plot_rmst_delta(ci, gene=gene, outname=f"Fig5_{gene}_RMSTdelta_ON_tau{int(TAU)}min_MAIN.pdf")


## Extended Data (ED) candidates

This section produces more detailed plots and QC summaries that are typically better suited for **Extended Data**:

- Full Kaplan–Meier curves for **ON** and **OFF** durations (all 4 groups).
- QC table: number of cells, number of runs, censoring fractions, mean/quantiles.

(Parametric model fits / mixture models can also be placed here if needed, but KM/RMST are usually sufficient for a clean story.)


In [ ]:
# --- ED: QC table + full KM curves (ON/OFF) ---

def qc_table(runs: pd.DataFrame, groups) -> pd.DataFrame:
    df = runs[runs["group"].isin(groups)].copy()

    qc_cells = (df[["group","cell_uid"]]
                .drop_duplicates()
                .groupby("group")["cell_uid"].nunique()
                .rename("n_cells")
                .reset_index())

    qc_runs = (df.groupby(["group","kind"])
               .agg(
                   n_runs=("length_min","size"),
                   frac_left=("left_censored","mean"),
                   frac_right=("right_censored","mean"),
                   mean_len=("length_min","mean"),
                   p90_len=("length_min", lambda x: np.percentile(x, 90)),
               )
               .reset_index())

    out = qc_runs.merge(qc_cells, on="group", how="left")
    return out.sort_values(["kind","group"])

for ds in datasets:
    gene = ds["gene"]
    runs = ds["runs"]

    qc = qc_table(runs, groups=GROUP_ORDER_FULL)
    display(qc)
    qc.to_csv(TABLEDIR / f"{gene}_QC_runs_cells_FULL.csv", index=False)

    # Full KM (exclude left-censored; include right-censored)
    plot_km_panel(runs, kind="ON",  gene=gene, groups=GROUP_ORDER_FULL, exclude_left=True, logy=True,
                  outname=f"ED_{gene}_KM_ON_FULL.pdf", title=f"$\\it{{{gene}}}$: KM active (exclude left-censored)")
    plot_km_panel(runs, kind="OFF", gene=gene, groups=GROUP_ORDER_FULL, exclude_left=True, logy=True,
                  outname=f"ED_{gene}_KM_OFF_FULL.pdf", title=f"$\\it{{{gene}}}$: KM inactive (exclude left-censored)")

## Optional: single-panel summary across genes

Nature main figures often benefit from a **single compact summary panel** that conveys the key message.

Below, we combine Nanog and Sox2 into one plot:

- x-axis: inhibitor condition (TSA, RGFP966 0h)
- y-axis: ΔRMST (ON, τ=30 min) vs DMSO
- points: gene (Nanog vs Sox2), with 95% CI from cell-cluster bootstrap


In [ ]:
# --- Combined ΔRMST plot (Nanog vs Sox2) ---

TAU = 30.0
BOOT_B = 2000
SEED = 0
GROUPS = GROUP_ORDER_MAIN  # change to FULL if desired

rows = []
for ds in datasets:
    gene = ds["gene"]
    ci = cluster_bootstrap_rmst_delta(
        ds["runs"], kind="ON", tau=TAU, base="DMSO",
        groups=GROUPS, B=BOOT_B, seed=SEED, exclude_left=True
    )
    if ci.empty:
        continue
    ci["gene"] = gene
    rows.append(ci)

if rows:
    comb = pd.concat(rows, ignore_index=True)
    display(comb)
    comb.to_csv(TABLEDIR / f"Combined_RMSTdelta_ON_tau{int(TAU)}min_MAIN.csv", index=False)

    # plot
    fig, ax = plt.subplots(figsize=PLOT_CFG["figsize_medium"])

    genes = comb["gene"].unique().tolist()
    inhibitors = [c.split("vs")[1].strip() for c in comb["compare"].unique().tolist()]
    inhibitors = [g for g in GROUPS if g != "DMSO" and g in inhibitors]

    x_base = np.arange(len(inhibitors))
    offsets = np.linspace(-0.12, 0.12, num=len(genes)) if len(genes) > 1 else [0.0]

    for off, gene in zip(offsets, genes):
        sub = comb[comb["gene"]==gene].copy()
        # align inhibitor order
        sub["grp"] = sub["compare"].apply(lambda s: s.split("vs")[1].strip())
        sub = sub.set_index("grp").loc[inhibitors].reset_index()

        x = x_base + off
        y = sub["Delta_obs(grp-base)"].values
        lo = sub["CI95_low"].values
        hi = sub["CI95_high"].values
        ax.errorbar(x, y, yerr=[y-lo, hi-y], fmt="o", capsize=3, label=gene)

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x_base)
    ax.set_xticklabels([nice_group(g) for g in inhibitors], rotation=30, ha="right")
    ax.set_ylabel("ΔRMST ON (min) vs DMSO (τ=30)")
    ax.set_title("ΔRMST ON summary (cell-cluster bootstrap 95% CI)")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False)

    savefig_pdf(fig, FIGDIR / "Fig5_Combined_RMSTdelta_ON_tau30min_MAIN.pdf")
else:
    print("[WARN] No datasets loaded; cannot make combined plot.")


## Optional: combined duty cycle summary across genes

A second simple cross-gene summary is the **duty cycle** (cell-level fraction of time ON),
which tends to be intuitive for readers.


In [ ]:
# --- Combined duty cycle plot (Nanog vs Sox2) ---

GROUPS = GROUP_ORDER_MAIN

rows = []
for ds in datasets:
    gene = ds["gene"]
    dc = duty_cycle_table(ds["perframe"], groups=GROUPS)
    dc["gene"] = gene
    rows.append(dc)

if rows:
    dc_all = pd.concat(rows, ignore_index=True)
    dc_all.to_csv(TABLEDIR / "Combined_duty_cycle_celllevel_MAIN.csv", index=False)

    fig, ax = plt.subplots(figsize=PLOT_CFG["figsize_wide"])

    # make positions: groups on x, genes as offsets
    genes = dc_all["gene"].unique().tolist()
    x_base = np.arange(len(GROUPS))
    offsets = np.linspace(-0.18, 0.18, num=len(genes)) if len(genes) > 1 else [0.0]

    # boxplot per (group, gene)
    pos = []
    data = []
    labels = []
    for gi, g in enumerate(GROUPS):
        for off, gene in zip(offsets, genes):
            sub = dc_all[(dc_all["group"]==g) & (dc_all["gene"]==gene)]["duty_cycle"].dropna().values
            if len(sub)==0:
                continue
            pos.append(gi + off)
            data.append(sub)
            labels.append(f"{nice_group(g)}\n{gene}")

    ax.boxplot(data, positions=pos, widths=0.25, showfliers=False)
    ax.set_xticks(x_base)
    ax.set_xticklabels([nice_group(g) for g in GROUPS], rotation=0)
    ax.set_ylabel("Duty cycle (fraction ON)")
    ax.set_title("Duty cycle (cell-level) across genes")
    ax.grid(axis="y", alpha=0.3)

    # legend-like gene labels
    # (simple text annotation)
    y0 = ax.get_ylim()[1]
    for off, gene in zip(offsets, genes):
        ax.text(x_base[-1] + 0.5, y0*(0.95 - 0.06*genes.index(gene)), gene)

    savefig_pdf(fig, FIGDIR / "Fig5_Combined_duty_cycle_MAIN.pdf")
else:
    print("[WARN] No datasets loaded; cannot make combined duty cycle plot.")


---

## Notes for figure assembly (Nature-style)

- The notebook saves **each panel as its own PDF**, so you can assemble multi-panel figures in Illustrator/Inkscape.
- Keep the main figure minimal (only the panels that advance the story).
- Place parameter-sensitivity checks (censoring, alternative fits, full 4-group comparisons) in **Extended Data**.

If you need a single multi-panel PDF output (e.g., `Fig5.pdf` with A–D laid out), we can add a final cell that uses `matplotlib.gridspec` to arrange panels.
